In [4]:
# Cell 1: Imports and Setup
import sys
import os
import matplotlib.pyplot as plt
import pandas as pd
import pypowsybl as pp
import pypowsybl.network as pn
import ipywidgets as widgets
from ipywidgets import interact
from OMPython import OMCSessionZMQ

# Define Paths
NOTEBOOK_NAME = "OM_test_3"
OUTPUT_DIR = os.path.join(os.getcwd(), f"{NOTEBOOK_NAME}_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[INFO] Outputs will be saved to: {OUTPUT_DIR}")

# Initialize OpenModelica Session
try:
    omc = OMCSessionZMQ()
    print("[SUCCESS] OpenModelica Session Started.")
    # Change OM working directory to keep things clean
    omc.sendExpression(f'cd("{OUTPUT_DIR}")')
except Exception as e:
    print(f"[ERROR] Could not start OpenModelica: {e}")

[INFO] Outputs will be saved to: /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_3_outputs
[SUCCESS] OpenModelica Session Started.


In [5]:
# Cell 2: [STEP 1] Load Libraries and User Model
import os

# 1. Clear session to start fresh
omc.sendExpression("clear()")

# 2. Load Modelica Standard Library (Essential)
print("[INFO] Loading Modelica Standard Library...")
if not omc.sendExpression("loadModel(Modelica)"):
    print("   [WARN] Could not load Modelica standard library via loadModel.")

# 3. Load the Dynawo Package
# VERIFICA QUE ESTA RUTA ES CORRECTA EN TU ORDENADOR
model_path = "/home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo"

if os.path.exists(model_path):
    print(f"[INFO] Loading package from: {model_path}")
    if not omc.sendExpression(f'loadFile("{model_path}")'):
        err = omc.sendExpression("getErrorString()")
        print(f"[ERROR] Failed to load package.mo: {err}")
    else:
        print("[SUCCESS] Dynawo package loaded.")
else:
    print(f"[ERROR] package.mo not found at {model_path}")

# 4. Define the Target Dynamic Model Name
# This must match the structure inside package.mo
DYNAMIC_MODEL_NAME = "Dynawo.Examples.BESS.WECC.MyBESS"

# Verify the class exists in memory
if omc.sendExpression(f"existClass({DYNAMIC_MODEL_NAME})"):
    print(f"[CHECK] Target model '{DYNAMIC_MODEL_NAME}' is ready.")
else:
    print(f"[ERROR] The class '{DYNAMIC_MODEL_NAME}' is NOT found in OpenModelica.")
    print("Loaded classes:", omc.sendExpression("getClassNames()"))

[INFO] Loading Modelica Standard Library...
[INFO] Loading package from: /home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo


[OMC log for 'sendExpression(loadFile("/home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo"), True)']: [scripting:warning:371] Dynawo requested package Modelica of version 3.2.3. Modelica 4.0.0 is used instead which states that it is only compatible with a conversion script. Use convertPackageToLibrary(Dynawo, Modelica, "4.0.0") to run the conversion script or proceed with potential issues as a result.


[SUCCESS] Dynawo package loaded.
[ERROR] The class 'Dynawo.Examples.BESS.WECC.MyBESS' is NOT found in OpenModelica.
Loaded classes: ('Dynawo', 'ModelicaServices', 'Complex', 'Modelica')


In [6]:
# Cell 3: [STEP 2] Generate Auxiliary Static Model (Global Wrapper)

# We create a simple wrapper in the Global Scope (no 'within' clause).
# This avoids namespace issues.
STATIC_MODEL_NAME = "MyBESS_Static_Global"
STATIC_FILE_PATH = os.path.join(OUTPUT_DIR, "MyBESS_Static_Global.mo")

# Note: We override the Generator to use the P_setpoint parameter
modelica_code = f"""
model {STATIC_MODEL_NAME}
  // We extend the user's dynamic model directly
  extends {DYNAMIC_MODEL_NAME}(
      // Force connection to Python parameters if they exist, 
      // otherwise we overwrite the components directly here.
      P_setpoint = P_setpoint_input
  );

  // Interface Parameter for Python
  parameter Real P_setpoint_input = 0.5;

  // We force the generator inside the model to take our setpoint
  // (Assuming your MyBESS.mo has a component named 'PRefPu' or 'GenPV')
  // If MyBESS.mo is the dynamic one, we need to trick it to act static:
  
  annotation(
      experiment(StopTime=0.0, Tolerance=1e-06)
  );
end {STATIC_MODEL_NAME};
"""

with open(STATIC_FILE_PATH, "w") as f:
    f.write(modelica_code)

print(f"[STEP 2] Generated global static wrapper: {STATIC_FILE_PATH}")

# Load the wrapper
if omc.sendExpression(f'loadFile("{STATIC_FILE_PATH}")'):
    print(f"[INFO] Loaded {STATIC_MODEL_NAME} successfully.")

    # Check it
    check_res = omc.sendExpression(f"checkModel({STATIC_MODEL_NAME})")
    if "Error" in check_res:
        print(f"[WARN] Check Model Warnings/Errors:\n{check_res}")
    else:
        print(f"[SUCCESS] {STATIC_MODEL_NAME} checks out ok.")
else:
    print(f"[ERROR] Could not load wrapper.")
    print(omc.sendExpression("getErrorString()"))

[STEP 2] Generated global static wrapper: /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_3_outputs/MyBESS_Static_Global.mo


[INFO] Loaded MyBESS_Static_Global successfully.


[OMC log for 'sendExpression(checkModel(MyBESS_Static_Global), True)']: [scripting:warning:371] Dynawo requested package Modelica of version 3.2.3. Modelica 4.0.0 is used instead which states that it is only compatible with a conversion script. Use convertPackageToLibrary(Dynawo, Modelica, "4.0.0") to run the conversion script or proceed with potential issues as a result.


OMCSessionException: [OMC log for 'sendExpression(checkModel(MyBESS_Static_Global), True)']: [translation:error:39] Base class Dynawo.Examples.BESS.WECC.MyBESS not found in scope MyBESS_Static_Global.

In [ ]:
# Cell 4: Define Static Network Context (Pure Python Math)
import math
import cmath


def get_network_boundary_conditions(p_setpoint_mw):
    """
    Calculates Voltage and Angle at the BESS connection point using
    standard complex impedance math (Ohm's Law), bypassing pypowsybl errors.
    """
    # --- Physical Parameters ---
    v_base_kv = 20.0  # Base Voltage at BESS
    v_grid_pu = 1.0  # Grid is infinite slack at 1.0 pu
    z_line_ohm = 0.1 + 1.0j  # Line Impedance (R + jX)

    # Grid Voltage (Phasor) in Volts
    v_grid_volts = complex(v_base_kv * 1000.0 * v_grid_pu, 0.0)

    # --- Solve for V_poi (Point of Interconnection) ---
    # We solve the equation: V_poi = V_grid + I * Z
    # Where I = (S_inj / V_poi)* # Since V_poi is on both sides, we iterate a few times (Fixed Point Iteration)

    v_poi = v_grid_volts  # Initial guess
    p_watts = p_setpoint_mw * 1e6
    q_vars = 0.0  # Assuming Unity Power Factor for initialization

    for _ in range(5):  # 5 iterations is plenty for convergence
        # Calculate Current injecting into grid: I = (S / V)*
        # Note: If P>0 (Generation), Current flows BESS -> Grid
        s_inj = complex(p_watts, q_vars)
        current = (s_inj / v_poi).conjugate()

        # KVL: V_bess = V_grid + Voltage_Drop
        # Since I flows BESS -> Grid, V_bess = V_grid + I * Z
        v_poi = v_grid_volts + (current * z_line_ohm)

    # --- Extract Results ---
    v_mag_pu = abs(v_poi) / (v_base_kv * 1000.0)
    v_angle_deg = math.degrees(cmath.phase(v_poi))

    return {
        "V_poi": float(v_mag_pu),
        "Angle_poi": float(v_angle_deg),
        "P_ref": float(p_setpoint_mw),
    }

In [ ]:
# Cell 5: Main Pipeline Function [STEPS 3, 4, 5]


def run_simulation_pipeline(p_setpoint_pu):
    # --- 0. Pre-calculation ---
    p_base = 100.0
    p_mw = p_setpoint_pu * p_base

    print(f"\n--- Starting Pipeline for P_setpoint = {p_setpoint_pu} pu ---")

    # Static Calc (Using Python Math from previous fix)
    bc = get_network_boundary_conditions(p_mw)  # Ensure you have the Python-only version here
    print(f"[INFO] Boundary Conditions: V={bc['V_poi']:.4f} pu")

    # ==============================================================================
    # [STEP 3] Run the STATIC model
    # ==============================================================================
    print("[STEP 3] Running Static Model...")

    # NOTE: We use the GLOBAL name defined in Cell 3
    # We pass P_setpoint_input which we defined in the wrapper
    sim_flags_static = (
        f"-override "
        f"Grid_V={bc['V_poi']},"
        f"Grid_Angle={bc['Angle_poi']},"
        f"P_setpoint_input={p_setpoint_pu}"  # Connecting to our wrapper param
    )

    # Simulate the wrapper
    omc.sendExpression(
        f'simulate({STATIC_MODEL_NAME}, stopTime=0.0, simflags="{sim_flags_static}")'
    )

    # ==============================================================================
    # [STEP 4] Extract INIT model output variables
    # ==============================================================================
    print("[STEP 4] Extracting Internal States...")

    # Update these variables based on what you actually need to initialize inside your BESS
    # Example: If your controller has an integrator named 'PI.x', you want 'PI.x'
    init_vars_to_extract = [
        "bess.converter.internal_angle",
        "bess.active_power_control.integrator.y",  # Example
        "bess.battery.SOC",  # Example
    ]

    static_res_file = f"{STATIC_MODEL_NAME}_res.mat"

    init_vars_to_extract = [
        "bess.battery.SOC",
        "bess.converter.internal_angle",
        # Add your specific vars here
    ]

    extracted_params = {}
    for var in init_vars_to_extract:
        val = omc.sendExpression(f'val({var}, 0.0, "{static_res_file}")')
        if val is not None:
            extracted_params[var] = val

    # ==============================================================================
    # [STEP 5] Run the DYNAMIC model
    # ==============================================================================
    print("[STEP 5] Running Dynamic Model...")

    # Now we simulate the ORIGINAL dynamic model (loaded in Cell 2)
    overrides = {
        "Grid_V": bc["V_poi"],
        "Grid_Angle": bc["Angle_poi"],
        "P_setpoint": p_setpoint_pu,
    }
    overrides.update(extracted_params)

    override_str = ",".join([f"{k}={v}" for k, v in overrides.items()])
    dynamic_res_file = f"{DYNAMIC_MODEL_NAME.split('.')[-1]}_res.mat"

    omc.sendExpression(
        f'simulate({DYNAMIC_MODEL_NAME}, stopTime=5.0, resultFile="{dynamic_res_file}", simflags="-override {override_str}")'
    )

    return dynamic_res_file

In [ ]:
# Cell 6: Visualization


def plot_results(res_file):
    if not os.path.exists(res_file):
        print("No result file found.")
        return

    # Helper to read variable from MAT file
    def get_val(var):
        return omc.sendExpression(f'readSimulationResult("{res_file}", {{{var}}})')[0]

    # Load data
    time = get_val("time")
    p_bess = get_val("bess.P_injection")  # Adjust variable name to your model
    q_bess = get_val("bess.Q_injection")  # Adjust variable name to your model
    v_term = get_val("bess.V_terminal")  # Adjust variable name to your model

    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    ax1.plot(time, p_bess, label="Active Power (MW)")
    ax1.plot(time, q_bess, label="Reactive Power (Mvar)")
    ax1.set_ylabel("Power")
    ax1.legend()
    ax1.grid(True)
    ax1.set_title(f"Dynamic Response: {res_file}")

    ax2.plot(time, v_term, label="Voltage (pu)", color="green")
    ax2.set_ylabel("Voltage (pu)")
    ax2.set_xlabel("Time (s)")
    ax2.legend()
    ax2.grid(True)

    plt.show()

In [ ]:
# Cell 7: Launch Interactive Widget


def on_change(p_setpoint):
    try:
        # Run the full 5-step pipeline
        res_file = run_simulation_pipeline(p_setpoint)

        # Only plot if simulation was successful (returned a valid path string)
        if res_file and isinstance(res_file, str) and os.path.exists(res_file):
            plot_results(res_file)
        else:
            print("[ERROR] Simulation pipeline did not return a valid result file.")

    except Exception as e:
        print(f"Workflow failed: {e}")


# Launch Widget
interact(
    on_change,
    p_setpoint=widgets.FloatSlider(
        value=0.5,
        min=-1.0,
        max=1.0,
        step=0.1,
        description="P Setpoint (pu)",
        continuous_update=False,
    ),
);